# Chapter 4: row count and key check of every Kestrel table

Each table is loaded, its grain printed and its key tested. A key that repeats is the finding: fix it before charting.
Synthetic data only.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA = Path("../data")   # run this notebook from the notebooks folder
def load(name, **kw):
    return pd.read_csv(DATA / f"{name}.csv", **kw)

In [ ]:
KEYS = {   # table: (key columns, one row is)
    "Brew": (["BrewId"], "one brewhouse run"),
    "Fill": (["FillId"], "one fermenter fill of two brews"),
    "Batch": (["FillSequence"], "one lager fill on the control chart"),
    "BatchAfterFix": (["FillSequence"], "one lager fill after the fix"),
    "HourlyGravity": (["FillId", "BatchHour"], "one fill, one whole hour from hour zero"),
    "Readings": (["Tank", "Tag", "Time"], "one tag reading on one tank"),
    "Lab": (["SampleId"], "one result for one analyte on one sample"),
    "Link": (["ChildId", "ParentId"], "one parent to child link"),
    "Run": (["RunId"], "one packaging run"),
    "Stop": (["RunId", "StopStart", "StopReason"], "one stop on the can line"),
    "Volume": (["Month"], "one month of cast and dispatch"),
    "Stage": (["Month", "Stage"], "one loss stage in one month"),
    "Daily": (["Date"], "one day of dispatch"),
    "Sales": (["Date", "Sku", "Account"], "one SKU, account and month"),
    "Depletions": (["Month", "Distributor"], "one distributor month"),
    "Sku": (["Sku"], "one SKU"),
    "Tank": (["Tank"], "one fermenter"),
    "Date": (["Date"], "one day"),
    "AsOf": (["Moment"], "one moment to read results as of"),
}
rows = []
for name, (key, grain) in KEYS.items():
    t = load(name)
    dup = int(t.duplicated(subset=key).sum())
    rows.append([name, len(t), " + ".join(key), "unique" if dup == 0 else f"{dup} repeats", grain])
report = pd.DataFrame(rows, columns=["table", "rows", "key", "key check", "one row is"])
print(report.to_string(index=False))

## A repeat, made on purpose
Join a lab result to minute readings by date and the one result lands on 1,440 rows.

In [ ]:
rd = load("Readings", parse_dates=["Time"])
temp = rd[(rd.Tag == "Temp") & (rd.Tank == "F07")].assign(Date=lambda d: d.Time.dt.date)
one_result = pd.DataFrame({"Date": [temp.Date.iloc[0]], "Value": [4.4]})
joined = temp.merge(one_result, on="Date")
print(len(joined), "rows now carry one lab result; a count based measure counts it", len(joined), "times")